In [ ]:
import pandas as pd
import geopandas as gpd
import glob
import warnings
warnings.filterwarnings('ignore')
pd.set_option("display.max_rows", 100)

import os
from dotenv import load_dotenv
import threading
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import requests
from pyproj import Transformer

DATA_PATH = r"D:\\02_사업관리\\03_분석컨설팅\\2026년\\20260902_도시계획과\\02_자료수집\\"
GPKG_PATH = DATA_PATH + r"2026-018_도시계획과.gpkg"
CACHE_PATH = DATA_PATH + "지오코딩_캐시_5179.csv"
load_dotenv("env")   # 노트북과 같은 폴더의 env
VWORLD_KEY = os.environ["VWORLD_API_KEY"]
VWORLD_URL = "https://api.vworld.kr/req/address"
MAX_WORKERS = 8
RETRY_FAILED = False    # True: 캐시에 실패(NaN)로 기록된 주소도 다시 요청
SAVE_EVERY = 500        # 요청 N건마다 캐시 저장

to_wgs84 = Transformer.from_crs("EPSG:5179", "EPSG:4326", always_xy=True)
session = requests.Session()
cache_lock = threading.Lock()

for year in range(2019, 2026):
    data_lists = glob.glob(DATA_PATH + rf"02_산업경제\\*_{str(year)}_*.csv", recursive=True)
    for data in data_lists:
        df = pd.read_csv(data, sep="|", dtype={'설립일자': str}, encoding="utf-16")
        df0 = df.copy()
        # 주소_지번 전처리
        addr = df0['주소_지번'].fillna('').str.split()
        is_gj = df0['주소_지번'].str.contains('기장군', na=False)
        df0['시도명'] = addr.str[0].replace({'부산시': '부산', '부산광역시': '부산'})
        df0['구군명'] = addr.str[1]
        df0['읍면동'] = addr.str[2].where(~is_gj, addr.str[2:4].str.join(' '))
        df0['번지수'] = (addr.str[3].where(~is_gj, addr.str[4]).str.replace('외', '번지', regex=True))
        df0['주소_지번'] = df0['시도명'] + ' ' + df0['구군명'] + ' ' + df0['읍면동'] + ' ' + df0['번지수']
        
        # 주소_도로명 전처리
        road = (df0['주소_도로명'].str.replace(r"\s+", ' ', regex=True).str.strip()
                           .str.replace(r'^기타', '', regex=True)
                           .str.extract(r'^(?P<시도>\S+) (?P<구군>\S+[구군]) (?:(?P<읍면>\S+[읍면]) )?'
                                        r'(?P<도로명>\S+(?:로|길)) ?(?P<지하>지하 )?(?P<건물번호>\d+(?:-\d+)?)?'))
        road['시도'] = road['시도'].replace({'부산시': '부산', '부산광역시': '부산'})
        df0['도로명'] = road['도로명']
        df0['건물번호'] = road['지하'].fillna('') + road['건물번호']
        df0['주소_도로명'] = (road['시도'] + ' ' + road['구군'] + ' ' + (road['읍면'] + ' ').fillna('')
                                + road['도로명'] + (' ' + df0['건물번호']).fillna(''))
        
        df0 = df0.dropna(subset=['주소_지번', '주소_도로명'])        
        df0 = df0[df.columns]


# 브이월드 지오코딩 이용
class QuotaError(Exception):
    """인증키 오류·일일 한도 초과 등 계속 진행할 수 없는 응답"""
    
    # ---------------------------------------------------------------------------
    # 캐시
    # ---------------------------------------------------------------------------
    def load_cache():
        if not os.path.exists(CACHE_PATH):
            return {}
        df = pd.read_csv(CACHE_PATH, encoding="utf-8-sig")
        if RETRY_FAILED:
            df = df.dropna(subset=["x", "y"])
        return {(r.주소, r.구분): (r.x, r.y) for r in df.itertuples(index=False)}

    def save_cache(cache):
        with cache_lock:
            rows = [(a, t, x, y) for (a, t), (x, y) in cache.items()]
        pd.DataFrame(rows, columns=["주소", "구분", "x", "y"]).to_csv(CACHE_PATH, index=False, encoding="utf-8-sig")

    # ---------------------------------------------------------------------------
    # 지오코딩
    # ---------------------------------------------------------------------------
    def request_vworld(addr, addr_type, retries=3):
        params = dict(service="address", request="getcoord", version="2.0", crs="EPSG:5179",
                    address=addr, refine="true", simple="false", format="json",
                    type=addr_type.lower(), key=VWORLD_KEY)
        for _ in range(retries):
            try:
                res = session.get(VWORLD_URL, params=params, timeout=10).json()["response"]
            except (requests.RequestException, ValueError, KeyError):
                continue
            status = res.get("status")
            if status == "OK":
                point = res["result"]["point"]
                return float(point["x"]), float(point["y"])
            if status == "NOT_FOUND":
                return float("nan"), float("nan")
            raise QuotaError(res.get("error", res))
        return None     # 네트워크 오류 - 캐시에 남기지 않음

    def geocode_all(targets, cache):
        """targets: {(주소, 'ROAD'|'PARCEL'), ...} 중 캐시에 없는 것만 요청"""
        todo = [k for k in targets if k not in cache]
        print(f"  요청 대상 {len(todo):,}건 (캐시 적중 {len(targets) - len(todo):,}건)")
        if not todo:
            return
        done = 0
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {ex.submit(request_vworld, a, t): (a, t) for a, t in todo}
            try:
                for fut in as_completed(futures):
                    result = fut.result()
                    if result is not None:
                        with cache_lock:
                            cache[futures[fut]] = result
                    done += 1
                    if done % SAVE_EVERY == 0:
                        save_cache(cache)
                        print(f"  {done:,}/{len(todo):,}")
            except QuotaError as e:
                for f in futures:
                    f.cancel()
                save_cache(cache)
                raise QuotaError(f"VWorld 오류로 중단(캐시 저장됨, 재실행 시 이어서 진행): {e}") from None
        save_cache(cache)

    def lookup(cache, addr, addr_type):
        if pd.isna(addr):
            return None
        xy = cache.get((addr, addr_type))
        if xy is None or pd.isna(xy[0]):
            return None
        return xy

    def _clean(s):
        s = s.str.replace(r"\s+", " ", regex=True).str.strip()
        return s.where(s.ne(""))

    # ---------------------------------------------------------------------------
    # 데이터프레임 좌표 부여
    # ---------------------------------------------------------------------------
    def add_coords(df, road_col="주소_도로명", parcel_col="주소_지번", cache=None):
        """전처리된 도로명·지번 주소로 좌표를 구해 x, y(EPSG:5179), 경도, 위도, 좌표_구분 컬럼을 추가한 사본을 반환.
        도로명으로 먼저 찾고, 실패한 건만 지번으로 다시 찾는다."""
        if cache is None:
            cache = load_cache()
            print(f"캐시 {len(cache):,}건 로드")
        road = _clean(df[road_col])
        parcel = _clean(df[parcel_col]).str.replace(r"번지$", "", regex=True)

        # 1차: 도로명
        geocode_all({(a, "ROAD") for a in road.dropna().unique()}, cache)
        road_xy = road.map(lambda a: lookup(cache, a, "ROAD"))

        # 2차: 도로명 실패 건만 지번
        need = road_xy.isna() & parcel.notna()
        geocode_all({(p, "PARCEL") for p in parcel[need].unique()}, cache)
        parcel_xy = parcel.where(need).map(lambda p: lookup(cache, p, "PARCEL"))

        xy = road_xy.where(road_xy.notna(), parcel_xy)
        out = df.copy()
        out["x"] = xy.str[0].astype(float)
        out["y"] = xy.str[1].astype(float)
        out["좌표_구분"] = np.where(road_xy.notna(), "ROAD", np.where(parcel_xy.notna(), "PARCEL", None))
        out["경도"], out["위도"] = to_wgs84.transform(out["x"].to_numpy(), out["y"].to_numpy())

        ok = out["x"].notna().sum()
        print(f"  성공 {ok:,}/{len(out):,} ({ok / len(out):.1%}) "
            f"- 도로명 {(out['좌표_구분'] == 'ROAD').sum():,}, 지번 {(out['좌표_구분'] == 'PARCEL').sum():,}")
        return out



             KEDCD 본점구분  사업장연번         사업자번호                  주소_지번  \
0               18   본점      1  1068103736.0   부산 부산진구 당감2동 857-7번지   
6              406  사업장      1  6018504903.0   부산 사하구 하단1동 619-22번지   
7              406  사업장      1  6018504903.0     부산 중구 중앙동4가 87-7번지   
8              478   본점      1  6048129827.0     부산 사상구 감전동 957-1번지   
10             638  사업장      1           NaN     부산 남구 감만1동 588-1번지   
...            ...  ...    ...           ...                    ...   
389253  9040129394   본점      1  4305300191.0        부산 서구 암남동 6-5번지   
389254  9040129464   본점      1  4211900534.0      부산 북구 만덕2동 74-4번지   
389255  9040129475   본점      1  4213200469.0   부산 사하구 하단2동 511-21번지   
389256  9040129480   본점      1  4214300354.0  부산 부산진구 부전2동 224-20번지   
389257  9040129484   본점      1  4214700444.0     부산 중구 중앙동4가 76-8번지   

                      주소_도로명 표준산업분류코드_10차      설립일자      매출_2019  고용_2019  \
0        부산 부산진구 신천대로 215-15       C31201  19681207    6080185.0     2